# Fine-tuning Qwen3-14B using PyTorch FSDP with PyTorch Lightning (Slurm)

This notebook shows how to fine-tune Qwen3-14B using PyTorch FSDP with [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) framework via **Slurm job scheduler**.

**Same configuration as Kubeflow Training Operator version** - reads the same `fine-tune.yaml`, `evaluate.yaml`, `convert-hf.yaml` files and generates sbatch scripts dynamically.

## Setup and Imports

In [ ]:
! pip install kubernetes
! pip install boto3
! pip install pyyaml

In [ ]:
import os
import subprocess
import sys
import yaml

# Set working directory
os.chdir(os.path.expanduser('~/amazon-eks-machine-learning-with-terraform-and-kubeflow'))
print(f"Working directory: {os.getcwd()}")

# Add src to path
src_dir = os.path.join(os.getcwd(), "src")
sys.path.insert(0, src_dir)

from k8s.utils import wait_for_helm_release_pods
from slurm.utils import generate_sbatch_from_yaml, wait_for_slurm_job, scale_slurm_nodeset, submit_slurm_job

# Paths to existing yaml configs (reused from Kubeflow example)
base_example_dir = os.path.join(os.getcwd(), 'examples', 'training', 'pytorch-lightning', 'qwen3-14b-sft')
slurm_notebook_dir = os.path.join(os.getcwd(), 'examples', 'training', 'slurm', 'pytorch-lightning', 'qwen3-14b-sft')

print(f"Base example dir: {base_example_dir}")
print(f"Slurm notebook dir: {slurm_notebook_dir}")

# Variables
release_name = 'ptl-qwen3-14b-sft'
namespace = 'slurm'

## Step 0: Build slurmd Container (One-time)

This step builds the slurmd container with CUDA, EFA, and PyTorch.
Skip if already built.

In [ ]:
# Check if image already exists
account_id = subprocess.run(
    ['aws', 'sts', 'get-caller-identity', '--query', 'Account', '--output', 'text'],
    capture_output=True, text=True
).stdout.strip()
region = 'us-west-2'
repo_name = 'slurmd-pytorch'
image_tag = '25.05.0-cu126-py312'
ecr_repo = f"{account_id}.dkr.ecr.{region}.amazonaws.com/{repo_name}"

result = subprocess.run(
    ['aws', 'ecr', 'describe-images', '--repository-name', repo_name,
     '--image-ids', f'imageTag={image_tag}', '--region', region],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(f"Image already exists: {ecr_repo}:{image_tag}")
    print("Skipping build. Delete the image to rebuild.")
else:
    print("Building slurmd container...")
    build_dir = os.path.join(os.getcwd(), 'eks-cluster', 'docker', 'slurmd')
    result = subprocess.run(['bash', 'build.sh'], cwd=build_dir, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise Exception("Container build failed")

print(f"slurmd image: {ecr_repo}:{image_tag}")

## Step 1: Download Qwen3-14B Model Weights

Replace `YourHuggingFaceToken` with your actual Hugging Face token.
(Same as Kubeflow flow - uses helm chart)

In [ ]:
hf_token = 'YourHuggingFaceToken'

cmd = [
    'helm', 'install', '--debug', release_name,
    'charts/machine-learning/model-prep/hf-snapshot',
    '--set-json', f'pvc=[{{"name":"slurm-fsx-pvc","mount_path":"/fsx"}},{{"name":"slurm-efs-pvc","mount_path":"/efs"}}]',
    '--set-json', f'env=[{{"name":"HF_MODEL_ID","value":"Qwen/Qwen3-14B"}},{{"name":"HF_TOKEN","value":"{hf_token}"}}]',
    '-n', namespace
]

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# Wait for model download to complete
wait_for_helm_release_pods(release_name, namespace)

In [ ]:
# Uninstall the model download job
cmd = ['helm', 'uninstall', release_name, '-n', namespace]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

## Step 2: Launch Fine-tuning via Slurm

Reads `fine-tune.yaml` + `slurm.yaml`, generates sbatch, and submits.

In [ ]:
# Load yaml configs
with open(f'{base_example_dir}/fine-tune.yaml') as f:
    base_config = yaml.safe_load(f)

with open(f'{base_example_dir}/slurm.yaml') as f:
    slurm_config = yaml.safe_load(f)

# Merge configs (slurm overlay takes precedence)
config = {**base_config, **slurm_config}

# Generate sbatch script
sbatch_content = generate_sbatch_from_yaml(config, release_name)
print("Generated sbatch script:")
print(sbatch_content[:1000] + "...")

In [ ]:
# Scale up Slurm compute nodes
num_nodes = config['resources']['nnodes']
scale_slurm_nodeset(num_nodes, namespace=namespace)

# Wait for slurmd pods to be ready
print("Waiting for slurmd pods...")
cmd = ['kubectl', 'wait', '--for=condition=Ready', 'pod', '-l', 'app.kubernetes.io/component=slurmd',
       '-n', namespace, '--timeout=600s']
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

In [ ]:
# Submit fine-tuning job
job_id = submit_slurm_job(sbatch_content, release_name, namespace=namespace)
print(f"Submitted fine-tuning job: {job_id}")

In [ ]:
# Wait for fine-tuning to complete
wait_for_slurm_job(job_id, namespace=namespace)

## Step 3: Evaluate Fine-tuned Model

In [ ]:
# Load and merge evaluate config
with open(f'{base_example_dir}/evaluate.yaml') as f:
    eval_config = yaml.safe_load(f)

eval_config = {**eval_config, **slurm_config}
eval_sbatch = generate_sbatch_from_yaml(eval_config, f"{release_name}-eval")

# Submit evaluation job
eval_job_id = submit_slurm_job(eval_sbatch, f"{release_name}-eval", namespace=namespace)
print(f"Submitted evaluation job: {eval_job_id}")

In [ ]:
# Wait for evaluation to complete
wait_for_slurm_job(eval_job_id, namespace=namespace)

## Step 4: Convert Fine-tuned Model to Hugging Face

In [ ]:
# Load and merge convert config
with open(f'{base_example_dir}/convert-hf.yaml') as f:
    convert_config = yaml.safe_load(f)

convert_config = {**convert_config, **slurm_config}
convert_sbatch = generate_sbatch_from_yaml(convert_config, f"{release_name}-convert")

# Submit conversion job
convert_job_id = submit_slurm_job(convert_sbatch, f"{release_name}-convert", namespace=namespace)
print(f"Submitted conversion job: {convert_job_id}")

In [ ]:
# Wait for conversion to complete
wait_for_slurm_job(convert_job_id, namespace=namespace)

In [ ]:
# Scale down compute nodes
scale_slurm_nodeset(0, namespace=namespace)
print("Scaled down compute nodes")

## Output

To access the output stored on EFS and FSx for Lustre file-systems:

```bash
kubectl apply -f eks-cluster/utils/attach-pvc.yaml -n kubeflow
kubectl exec -it -n kubeflow attach-pvc -- /bin/bash
```

### Logs
Training logs are available in `/efs/home/ptl-qwen3-14b-sft/logs` folder.

### Output
Training output are available in `/efs/home/ptl-qwen3-14b-sft/output` folder.